# 030_polyp_yolo_sahi
## YOLOv12x (SAHI) + SAM2

Uses Slicing Aided Hyper Inference (SAHI) to tile the image into 512x512 overlapping patches, improving small polyp detection before SAM2 refinement.

**Self-contained**: This notebook has zero external file dependencies. Any user can run it with their own Google Drive and Colab account, provided they have the polyp dataset and YOLO weights at the paths specified below.


In [ ]:
# Cell 1: Environment Setup
!pip install -q ultralytics sahi
from google.colab import drive
import time
for _ in range(5):
    try:
        drive.mount('/content/drive', force_remount=True)
        break
    except Exception as e:
        print('Mount failed, retrying...', e)
        time.sleep(5)


## Configuration

Edit these paths if your data is in a different location:
- **Dataset**: `/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset/`
- **YOLO Weights**: `/content/drive/MyDrive/models/trained/025_polyp_yolov12x_aug/train/weights/best.pt`
- **SAM2 Weights**: `sam2_b.pt` (auto-downloaded by ultralytics)
- **Confidence Threshold**: 0.25


In [ ]:
# Cell 3: Evaluate — YOLOv12x (SAHI) + SAM2
# Uses Slicing Aided Hyper Inference (SAHI) to tile the image into 512x512 overlapping patches, improving small polyp detection before SAM2 refinement.

import os
import glob
from pathlib import Path
import numpy as np
import cv2
from tqdm import tqdm
from ultralytics import SAM
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

def yolo_to_mask(label_path, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)
    if not os.path.exists(label_path):
        return mask
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6: continue
            coords = list(map(float, parts[1:]))
            pts = np.array(coords).reshape(-1, 2)
            pts[:, 0] *= w
            pts[:, 1] *= h
            cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
    return mask

def main():
    # --- Hardcoded Config (no external dependencies) ---
    YOLO_WEIGHTS = "/content/drive/MyDrive/models/trained/025_polyp_yolov12x_aug/train/weights/best.pt"
    SAM_WEIGHTS = "sam2_b.pt"
    TEST_IMAGES_DIR = "/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset/test/images"
    TEST_LABELS_DIR = "/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset/test/labels"
    CONF = 0.25

    # Initialize YOLO via SAHI Wrapper
    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics',
        model_path=YOLO_WEIGHTS,
        confidence_threshold=CONF,
        device="cuda:0" if __import__("torch").cuda.is_available() else "cpu", # Use GPU in Colab
    )

    sam_model = SAM(SAM_WEIGHTS)

    test_images = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, '*.*')))
    test_labels_dir = TEST_LABELS_DIR
    
    output_dir = Path('/tmp/exploration_output')
    output_dir.mkdir(parents=True, exist_ok=True)

    ious = []
    dices = []

    for img_path in tqdm(test_images, desc="Evaluating YOLO (SAHI) -> SAM2"):
        img = cv2.imread(img_path)
        if img is None: continue
        h, w = img.shape[:2]

        label_path = os.path.join(test_labels_dir, Path(img_path).stem + ".txt")
        gt_mask = yolo_to_mask(label_path, h, w)

        # 1. YOLO predicts boxes using Slicing Aided Hyper Inference
        result = get_sliced_prediction(
            img_path, # passing path is safer for SAHI
            detection_model,
            slice_height=512,
            slice_width=512,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2,
            verbose=0
        )
        object_predictions = result.object_prediction_list
        boxes = [obj.bbox.to_xyxy() for obj in object_predictions]
        
        pred_mask = np.zeros((h, w), dtype=np.uint8)
        if len(boxes) > 0:
            # 2. SAM2 refines masks using YOLO boxes as prompt
            for box in boxes:
                sam_results = sam_model(img, bboxes=box, verbose=False)[0]
                if sam_results.masks is not None:
                    mask_data = sam_results.masks.data.cpu().numpy()
                    if mask_data.ndim == 3: mask_data = mask_data[0] # Take first instance
                    if mask_data.shape != (h, w):
                        mask_data = cv2.resize(mask_data.astype(np.float32), (w, h), interpolation=cv2.INTER_NEAREST)
                    pred_mask = np.logical_or(pred_mask, mask_data > 0).astype(np.uint8)

        # 3. Calculate metrics
        intersection = np.logical_and(gt_mask, pred_mask).sum()
        union = np.logical_or(gt_mask, pred_mask).sum()
        
        if union == 0:
            iou = 1.0 # True negative
        else:
            iou = intersection / union

        # Dice score
        denom = gt_mask.sum() + pred_mask.sum()
        if denom == 0:
            dice = 1.0
        else:
            dice = 2 * intersection / denom

        ious.append(iou)
        dices.append(dice)
        
    mIoU = np.mean(ious)
    mDice = np.mean(dices)
    print(f"Testing completed.")
    print(f"mIoU: {mIoU:.4f}")
    print(f"mDice: {mDice:.4f}")
    
    # Save results
    with open(output_dir / "metrics.txt", "w") as f:
        f.write(f"mIoU: {mIoU:.4f}\n")
        f.write(f"mDice: {mDice:.4f}\n")

main()


In [ ]:
# Cell 4: Cleanup GPU resources
from google.colab import runtime
runtime.unassign()
